# Daily Challenge: Breast Cancer Prediction

## 1. Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Load the Breast Cancer Wisconsin dataset (equivalent to the Kaggle version)
raw = load_breast_cancer()
df = pd.DataFrame(raw.data, columns=raw.feature_names)
df.insert(1, "diagnosis", pd.Categorical.from_codes(raw.target, ["M", "B"]))

print("Shape:", df.shape)
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Drop unnecessary columns (none here, but we drop 'id' if it were present)
# The sklearn version has no id column; this step is kept for completeness
print("Columns after cleaning:", df.columns.tolist())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="diagnosis", data=df, palette="magma")
plt.title("Diagnosis Count (B = Benign, M = Malignant)")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 2. Data Preprocessing

In [ ]:
# Counts of unique values in the diagnosis column
print("Unique value counts in 'diagnosis':")
print(df["diagnosis"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Map categorical values to numerical: M -> 1, B -> 0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

print("Diagnosis after mapping:")
print(df["diagnosis"].value_counts())

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\nTraining set: {X_train_sc.shape}")
print(f"Test set:     {X_test_sc.shape}")

## 3. Model Building and Evaluation

### 3.1 Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(max_iter=10000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Accuracy: {acc_lr:.4f}")
print(classification_report(y_test, y_pred_lr, target_names=["Benign", "Malignant"]))

### 3.2 K-Nearest Neighbours

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)

acc_knn = accuracy_score(y_test, y_pred_knn)
print(f"KNN Accuracy: {acc_knn:.4f}")
print(classification_report(y_test, y_pred_knn, target_names=["Benign", "Malignant"]))

### 3.3 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {acc_rf:.4f}")
print(classification_report(y_test, y_pred_rf, target_names=["Benign", "Malignant"]))

### 3.4 Support Vector Machine

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm.fit(X_train_sc, y_train)
y_pred_svm = svm.predict(X_test_sc)

acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"SVM Accuracy: {acc_svm:.4f}")
print(classification_report(y_test, y_pred_svm, target_names=["Benign", "Malignant"]))

## 4. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN", "Random Forest", "SVM"],
    "Accuracy": [acc_lr, acc_knn, acc_rf, acc_svm]
}).sort_values("Accuracy", ascending=False).reset_index(drop=True)

print(results.to_string(index=False))

plt.figure(figsize=(7, 4))
bars = plt.barh(results["Model"], results["Accuracy"], color="steelblue")
plt.xlim(0, 1)
plt.xlabel("Accuracy")
plt.title("Model Accuracy Comparison")
for bar, val in zip(bars, results["Accuracy"]):
    plt.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
             f"{val:.4f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

best_model = results.iloc[0]
print(f"\nBest model: {best_model['Model']} with accuracy {best_model['Accuracy']:.4f}")

In [ ]:
predictions = {
    "Logistic Regression": y_pred_lr,
    "KNN": y_pred_knn,
    "Random Forest": y_pred_rf,
    "SVM": y_pred_svm
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, preds) in zip(axes, predictions.items()):
    sns.heatmap(confusion_matrix(y_test, preds), annot=True, fmt="d",
                cmap="magma", cbar=False, ax=ax,
                xticklabels=["Benign", "Malignant"],
                yticklabels=["Benign", "Malignant"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()